In [0]:
sale_path = "/Volumes/workspace/default/learning/sales.csv"

In [0]:
sale = spark.read.csv(sale_path, header = True, inferSchema = True)

In [0]:
display(sale)

In [0]:
from pyspark.sql.functions import col, count, sum,round, avg, max, min, countDistinct, lit

In [0]:
co = sale.groupBy(col("category")).agg(
    sum(col("quantity") * col("unit_price")).alias("total_sales"),
    count(col("category")).alias("category_count")
)
display(co)

In [0]:
%sql
select category,count(*) as ordercount ,round(sum(unit_price*quantity),2) as total_revenue
from read_files('/Volumes/workspace/default/learning/sales.csv',
                format =>'csv',
                header=>'true',
                inferSchema=>'true'
)
group by category
order by total_revenue desc

In [0]:
cate = sale.filter(col("category")== "Electronics")
display(cate)

In [0]:
%sql
select * from read_files('/Volumes/workspace/default/learning/sales.csv',
                format =>'csv',
                header=>'true',
                inferSchema=>'true'
)
where category = "Electronics"             

In [0]:
%sql
select * from read_files('/Volumes/workspace/default/learning/sales.csv',
                format =>'csv',
                header=>'true',
                inferSchema=>'true'
)
where category = "Electronics"
and (quantity * unit_price) > 20

In [0]:
cate1 = sale.filter((col("category") == "Electronics") & ((col("quantity") * col("unit_price")) > 50))
display(cate1)

In [0]:
date = sale.filter((col("order_date")<= '2024-10-10') & (col("order_date") >= '2024-10-01')).show()

In [0]:
%sql
-- Rank products by revenue within each category
SELECT 
    category,
    product_name,
    quantity * unit_price as revenue,
    RANK() OVER (PARTITION BY category ORDER BY quantity * unit_price DESC) as rank
FROM read_files('/Volumes/workspace/default/learning/sales.csv',
                format =>'csv',
                header=>'true',
                inferSchema=>'true')

In [0]:
%sql
-- Rank products by revenue within each category
select product_name , round(sum(quantity*unit_price),2) as revenue , rank()over(order by sum(quantity*unit_price)desc) as rank
from read_files('/Volumes/workspace/default/learning/sales.csv',
                format =>'csv',
                header=>'true',
                inferSchema=>'true'
)
group by product_name
order by revenue desc

Which product had the highest single-order revenue?

In [0]:
# high = sale.filter(col("quantity") * col("unit_price")).orderBy(col("product_name"))
# high.show()

In [0]:
%sql
select product_name, sum(unit_price * quantity) as revenue
from read_files('/Volumes/workspace/default/learning/sales.csv',
                format =>'csv',
                header=>'true',
                inferSchema=>'true')
group by product_name
order by revenue desc
limit 1;

In [0]:
sale.show(5)

What's the average order value (quantity × unit_price) across all orders?

In [0]:
%sql
select avg(unit_price * quantity) as avgor
from read_files('/Volumes/workspace/default/learning/sales.csv',
                format =>'csv',
                header=>'true',
                inferSchema=>'true')

How many orders were placed for each product category, ordered by count?

In [0]:
%sql
select category, count(*) as count
from read_files('/Volumes/workspace/default/learning/sales.csv',
                format =>'csv',
                header=>'true',
                inferSchema=>'true')
                
group by category
order by count desc